This notebook prcesses raw skeletons by breaking branches, and remerging the fragments. After all strips from a given directory are processed, they are then combined using translation and offset information.

Note: Will voxel anisotropy need to be accounted for before applying an affine transform to the skeletons?

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import navis
from joblib import dump, load
import os
import requests
import glob
from cloudvolume import CloudVolume, Skeleton
import numpy as np
from pathlib import Path

from ac_segmentation.reconnect_stack_navis import reconnect, read_navis_neurons_tar, write_navis_skels_tar, swap_dimensions, apply_transform_skeletons, remove_translate_nodes
from ac_segmentation.postprocess import  combine_skels_strips
import uuid

from scipy import ndimage as ndi
import numpy as np
import pandas as pd

In [ ]:
#declare directories and model file paths
skel_dir = "/ACdata/Users/connorl/Skeletons/Test/Raw_Skels/"
out_dir = "/ACdata/Users/connorl/Skeletons/Test/Processed_Skels/"
sc = load("/ACdata/Users/connorl/Models/scaler.joblib") #scaler file
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib") #model file

os.makedirs(skel_dir, exist_ok = True)
os.makedirs(out_dir, exist_ok = True) 

In [ ]:
###Post-Process individual strips before combining 
files = glob.glob(skel_dir + "*.gz")
affines = []
for ind,file in enumerate(files):
    #break and reconnect skeletons
    skels, merges = reconnect(in_skels = file, cl = cl, sc = sc, min_nodes = 20, query_dis = 10, min_collin=.5)

    #swap x and z dimensions, as the optimal dimension orientation for segmentation is zyx
    skels = swap_dimensions(skels)

    #optionally apply affine transformations
    #skels = apply_transform_skeletons(skels, affine[ind])

    #write to tar file
    name = file.split("/")[-1]
    write_navis_skels_tar(out_dir + "Rec_" + name, skels)

In [ ]:
###Combine skeletons of multiple strips in directory
files = glob.glob(out_dir + "*.gz")
files.sort()
out_sk = []
tx,ty,tz=[0,-246,0]
bound_box=[0,22848,0,42,0,288]

for ind,file in enumerate(files):
    skels = read_navis_neurons_tar(file)
    #skip last file, as node removal is unnecessary 
    if ind != len(files)-1:
        sk = remove_translate_nodes(skels, trans=[tx*ind,ty*ind,tz*ind], bound_box=bound_box)
        out_sk += sk
    else:
        sk = remove_translate_nodes(skels, trans=[tx*ind,ty*ind,tz*ind], bound_box=[0,0,0,0,0,0])
        out_sk += sk

out_sk = navis.NeuronList(out_sk)

#connect skeletons between strips
skels_comb, merges = reconnect(in_skels = out_sk, cl = cl, sc = sc, min_nodes = 20, query_dis = 10, min_collin=.5, resample=None, smooth=None)
write_navis_skels_tar(out_dir + "Combined_Skels.swc.gz", skels_comb)